# Alpha: TD Setup (9-bar)

> DeMark's 9-bar setup identifies exhaustion: nine consecutive closes each below the close four bars earlier. When the count completes the market has moved too far too fast and a reversal is probable.

## Notebook structure
| Section | Description |
|---------|-------------|
| 0 | Install & imports |
| 1 | Config & data |
| 2 | Helper functions |
| 3 | TD Sequential engine |
| 4 | V1 signal generation — conventional & contrarian polarity |
| 5 | Returns & performance |
| 6 | Per-ticker breakdown |
| 7 | Parameter sensitivity (setup count 7–11) |
| 8 | Representative equity curves |
| 9 | Export signals for td-swarm |

**Run all cells top-to-bottom in a fresh Colab runtime.**


In [ ]:
!pip install yfinance --quiet
print("Dependencies installed.")

In [ ]:
# ── Google Drive: optional mirror (skipped in CI) ─────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/swarm_alpha_streams'
    import os; os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f'Drive mirror: {DRIVE_BASE}')
except Exception:
    DRIVE_BASE = None


## Section 1 — Config & Data

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time

TICKERS    = ['SPY', 'QQQ', 'IWM', 'TLT', 'HYG',
              'GLD', 'USO', 'UUP', 'EEM', 'VNQ',
              'SLV', 'XLF', 'XLE']
START_DATE = '2015-01-01'
COST_BPS   = 5


SETUP_COUNT   = 9
TDST_WINDOW   = 9

print("Fetching OHLCV...")
raw = yf.download(
    TICKERS,
    start       = START_DATE,
    auto_adjust = True,
    progress    = False
)

close   = raw['Close'].dropna(how='all').ffill(limit=3)
high    = raw['High'].dropna(how='all').ffill(limit=3)
low     = raw['Low'].dropna(how='all').ffill(limit=3)
returns = close.pct_change()

print(f"  Shape : {close.shape}")
print(f"  Range : {close.index[0].date()} -> "
      f"{close.index[-1].date()}")
print("Data ready")




## Section 2 — Helper Functions

In [ ]:
def compute_atr(high, low, close, period=14):
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low  - close.shift(1)).abs()
    tr  = pd.concat(
        [tr1, tr2, tr3], axis=1,
        keys=['a', 'b', 'c']
    ).groupby(level=1, axis=1).max()
    return tr.rolling(period).mean()


def apply_costs(returns, position, cost_bps=5):
    cost     = cost_bps / 10_000
    turnover = position.diff().abs()
    return returns * position - turnover * cost


def performance_summary(r, name=''):
    r       = r.dropna()
    cum     = (1 + r).cumprod()
    ann     = 252
    ann_ret = (1 + r.mean()) ** ann - 1
    ann_vol = r.std() * np.sqrt(ann)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan
    roll_max = cum.cummax()
    mdd     = (cum / roll_max - 1).min()
    calmar  = ann_ret / abs(mdd) if mdd != 0 else np.nan
    hits    = (r > 0).mean()
    return pd.Series({
        'Ann. Return': ann_ret,
        'Ann. Vol':    ann_vol,
        'Sharpe':      sharpe,
        'Max DD':      mdd,
        'Calmar':      calmar,
        'Hit Rate':    hits,
    }, name=name)


print("Helpers loaded")




## Section 3 — TD Sequential Engine

In [ ]:
#
# All count lengths passed as arguments.
# Canonical DeMark values are the defaults.
#
# setup_count        : consecutive bars to complete setup (default 9)
# countdown_count    : countdown bars to fire signal    (default 13)
# setup_lookback     : close < close[-N] lookback       (default 4)
# countdown_lookback : close <= low[-N] lookback        (default 2)


def rolling_consecutive_count(cond: pd.DataFrame) -> pd.DataFrame:
    """
    Convert binary condition DataFrame into a consecutive-count
    DataFrame that resets to zero on False.
    """
    counts = pd.DataFrame(
        0, index=cond.index, columns=cond.columns)
    for col in cond.columns:
        s         = cond[col]
        group_id  = (s == 0).cumsum()
        raw_count = s.groupby(group_id).cumsum()
        counts[col] = raw_count * s
    return counts


def compute_td_setup_param(
        close: pd.DataFrame,
        high: pd.DataFrame,
        low: pd.DataFrame,
        setup_count: int    = 9,
        setup_lookback: int = 4) -> dict:
    """
    Parameterised TD Setup.

    Bullish: close < close[setup_lookback bars ago] for
             setup_count consecutive bars
    Bearish: close > close[setup_lookback bars ago] for
             setup_count consecutive bars

    Returns:
        bull_count        : running consecutive count
        bear_count        : running consecutive count
        bull_setup_signal : fires on bar == setup_count
        bear_setup_signal : fires on bar == setup_count
        bull_perfect      : perfect bullish setup flag
        bear_perfect      : perfect bearish setup flag
    """
    close_n = close.shift(setup_lookback)

    bull_cond = (close < close_n).astype(int)
    bear_cond = (close > close_n).astype(int)

    bull_count = rolling_consecutive_count(bull_cond)
    bear_count = rolling_consecutive_count(bear_cond)

    bull_signal = (bull_count == setup_count).astype(int)
    bear_signal = (bear_count == setup_count).astype(int)

    return {
        'bull_count':        bull_count,
        'bear_count':        bear_count,
        'bull_setup_signal': bull_signal,
        'bear_setup_signal': bear_signal,
    }


def compute_tdst(
        close: pd.DataFrame,
        high: pd.DataFrame,
        low: pd.DataFrame,
        bull_setup_signal: pd.DataFrame,
        bear_setup_signal: pd.DataFrame,
        setup_count: int = 9) -> dict:
    """
    TDST Support and Resistance levels.
    Kept as a separate function for modularity — allows
    TDST window to be varied independently of the setup
    count if needed.

    TDST Support    : lowest low of completed bullish setup
    TDST Resistance : highest high of completed bear setup

    Returns:
        tdst_support     : price level, forward-filled
        tdst_resistance  : price level, forward-filled
        tdst_long_valid  : close > tdst_support
        tdst_short_valid : close < tdst_resistance
    """
    lowest_n  = low.rolling(setup_count).min()
    highest_n = high.rolling(setup_count).max()

    tdst_support = pd.DataFrame(
        np.nan, index=close.index, columns=close.columns)
    tdst_resistance = pd.DataFrame(
        np.nan, index=close.index, columns=close.columns)

    tdst_support[bull_setup_signal == 1]    = \
        lowest_n[bull_setup_signal == 1]
    tdst_resistance[bear_setup_signal == 1] = \
        highest_n[bear_setup_signal == 1]

    tdst_support    = tdst_support.ffill()
    tdst_resistance = tdst_resistance.ffill()

    tdst_long_valid  = (close > tdst_support).astype(int)
    tdst_short_valid = (close < tdst_resistance).astype(int)

    return {
        'tdst_support':     tdst_support,
        'tdst_resistance':  tdst_resistance,
        'tdst_long_valid':  tdst_long_valid,
        'tdst_short_valid': tdst_short_valid,
    }


def compute_td_countdown_param(
        close: pd.DataFrame,
        high: pd.DataFrame,
        low: pd.DataFrame,
        bull_setup_signal: pd.DataFrame,
        bear_setup_signal: pd.DataFrame,
        countdown_count: int    = 13,
        countdown_lookback: int = 2) -> dict:
    """
    Parameterised TD Countdown with recycling.

    Fire-once logic: CD signal fires exactly once on the bar
    count reaches countdown_count, then deactivates. This
    keeps entry and hold as separate concerns — entry is
    the discrete event here; carry_signal manages the hold.

    Re-entry signal (bull_cd_reentry / bear_cd_reentry):
    fires when a RECYCLE completes a new countdown. This
    is the pyramiding add signal — only meaningful when
    pyramiding=True in carry_signal.

    Bullish condition : close <= low[countdown_lookback bars ago]
    Bearish condition : close >= high[countdown_lookback bars ago]

    Recycling  : same-direction setup resets the countdown
    Cancellation: opposing setup cancels the countdown
    """
    low_n  = low.shift(countdown_lookback)
    high_n = high.shift(countdown_lookback)

    bull_cd_cond = (close <= low_n)
    bear_cd_cond = (close >= high_n)

    # Initialise output DataFrames
    bull_cd_count   = pd.DataFrame(0, index=close.index, columns=close.columns)
    bear_cd_count   = pd.DataFrame(0, index=close.index, columns=close.columns)
    bull_cd_active  = pd.DataFrame(0, index=close.index, columns=close.columns)
    bear_cd_active  = pd.DataFrame(0, index=close.index, columns=close.columns)
    bull_cd_signal  = pd.DataFrame(0, index=close.index, columns=close.columns)
    bear_cd_signal  = pd.DataFrame(0, index=close.index, columns=close.columns)
    bull_cd_reentry = pd.DataFrame(0, index=close.index, columns=close.columns)
    bear_cd_reentry = pd.DataFrame(0, index=close.index, columns=close.columns)

    bull_recycle_counts = {}
    bear_recycle_counts = {}

    n_bars = len(close)

    for col in close.columns:

        # -- Bullish countdown ----------------------------
        in_cd       = False
        b_count     = 0
        b_was_reset = False    # True after a recycle, until next completion
        b_counts    = [0] * n_bars
        b_active    = [0] * n_bars
        b_signal    = [0] * n_bars
        b_reentry   = [0] * n_bars
        b_recycles  = 0

        for i in range(n_bars):

            # New bullish setup — recycle if already in countdown
            if bull_setup_signal[col].iloc[i] == 1:
                if in_cd:
                    b_recycles += 1
                    b_was_reset = True   # next completion = re-entry
                b_count = 0
                in_cd   = True

            # Opposing setup cancels countdown entirely
            if bear_setup_signal[col].iloc[i] == 1:
                in_cd       = False
                b_count     = 0
                b_was_reset = False

            # Accumulate count while active and incomplete
            if in_cd and b_count < countdown_count:
                if bull_cd_cond[col].iloc[i]:
                    b_count += 1

            # Fire once on reaching countdown_count, then deactivate
            if in_cd and b_count == countdown_count:
                if b_was_reset:
                    b_reentry[i] = 1     # pyramiding add signal
                else:
                    b_signal[i]  = 1     # initial entry signal
                in_cd       = False      # deactivate — won't re-fire
                b_count     = 0
                b_was_reset = False

            b_counts[i] = b_count if in_cd else 0
            b_active[i] = 1       if in_cd else 0

        bull_cd_count[col]       = b_counts
        bull_cd_active[col]      = b_active
        bull_cd_signal[col]      = b_signal
        bull_cd_reentry[col]     = b_reentry
        bull_recycle_counts[col] = b_recycles

        # -- Bearish countdown ----------------------------
        in_cd       = False
        r_count     = 0
        r_was_reset = False
        r_counts    = [0] * n_bars
        r_active    = [0] * n_bars
        r_signal    = [0] * n_bars
        r_reentry   = [0] * n_bars
        r_recycles  = 0

        for i in range(n_bars):

            if bear_setup_signal[col].iloc[i] == 1:
                if in_cd:
                    r_recycles += 1
                    r_was_reset = True
                r_count = 0
                in_cd   = True

            if bull_setup_signal[col].iloc[i] == 1:
                in_cd       = False
                r_count     = 0
                r_was_reset = False

            if in_cd and r_count < countdown_count:
                if bear_cd_cond[col].iloc[i]:
                    r_count += 1

            if in_cd and r_count == countdown_count:
                if r_was_reset:
                    r_reentry[i] = 1
                else:
                    r_signal[i]  = 1
                in_cd       = False
                r_count     = 0
                r_was_reset = False

            r_counts[i] = r_count if in_cd else 0
            r_active[i] = 1       if in_cd else 0

        bear_cd_count[col]       = r_counts
        bear_cd_active[col]      = r_active
        bear_cd_signal[col]      = r_signal
        bear_cd_reentry[col]     = r_reentry
        bear_recycle_counts[col] = r_recycles

    return {
        'bull_cd_count':   bull_cd_count,
        'bear_cd_count':   bear_cd_count,
        'bull_cd_signal':  bull_cd_signal,   # fires exactly once per countdown
        'bear_cd_signal':  bear_cd_signal,
        'bull_cd_reentry': bull_cd_reentry,  # fires on recycle completion
        'bear_cd_reentry': bear_cd_reentry,
        'bull_cd_active':  bull_cd_active,
        'bear_cd_active':  bear_cd_active,
        'bull_recycled':   bull_recycle_counts,
        'bear_recycled':   bear_recycle_counts,
    }


# -- Run the engine ----------------------------------------
print("Running TD Sequential engine...")
print("(Countdown loop per ticker — expect 5-15 seconds)\n")

t0 = time.time()

td_setup = compute_td_setup_param(
    close, high, low,
    setup_count=9, setup_lookback=4)

td_tdst = compute_tdst(
    close, high, low,
    td_setup['bull_setup_signal'],
    td_setup['bear_setup_signal'],
    setup_count=9)

td_countdown = compute_td_countdown_param(
    close, high, low,
    td_setup['bull_setup_signal'],
    td_setup['bear_setup_signal'],
    countdown_count=13, countdown_lookback=2)

elapsed = time.time() - t0
print(f"  Done in {elapsed:.1f}s\n")

# -- Diagnostics -------------------------------------------
bull_9s  = td_setup['bull_setup_signal'].sum().sum()
bear_9s  = td_setup['bear_setup_signal'].sum().sum()
bull_13s = td_countdown['bull_cd_signal'].sum().sum()
bear_13s = td_countdown['bear_cd_signal'].sum().sum()
bull_re  = td_countdown['bull_cd_reentry'].sum().sum()
bear_re  = td_countdown['bear_cd_reentry'].sum().sum()

print(f"  Bull setup signals  : {int(bull_9s)}")
print(f"  Bear setup signals  : {int(bear_9s)}")
print(f"  Bull CD13 signals   : {int(bull_13s)}")
print(f"  Bear CD13 signals   : {int(bear_13s)}")
print(f"  Bull reentry signals: {int(bull_re)}")
print(f"  Bear reentry signals: {int(bear_re)}")

print(f"\n  Sanity (CD13s <= Setup9s):")
print(f"  Bull ratio: {bull_13s/max(bull_9s,1):.2f}  (expect <= 1.0)")
print(f"  Bear ratio: {bear_13s/max(bear_9s,1):.2f}  (expect <= 1.0)")

print(f"\n  Recycling events per ticker:")
print(f"  {'Ticker':8s}  {'Bull':6s}  {'Bear':6s}")
for ticker in TICKERS:
    br = td_countdown['bull_recycled'][ticker]
    rr = td_countdown['bear_recycled'][ticker]
    print(f"  {ticker:8s}  {br:6d}  {rr:6d}")

print("\nTD engine complete")




In [ ]:
def carry_signal(entry: pd.DataFrame,
                 gate: pd.DataFrame,
                 max_hold: int,
                 pyramiding: bool = False,
                 reentry: pd.DataFrame = None,
                 max_layers: int = 3) -> pd.DataFrame:
    pos = pd.DataFrame(
        0.0,
        index=entry.index,
        columns=entry.columns)

    for col in entry.columns:
        col_idx = pos.columns.get_loc(col)

        if not pyramiding:
            hold = 0
            for i in range(len(entry)):
                if entry[col].iloc[i] == 1:
                    hold = max_hold
                if gate[col].iloc[i] == 0:
                    hold = 0
                pos.iloc[i, col_idx] = 1.0 if hold > 0 else 0.0
                if hold > 0:
                    hold -= 1

        else:
            layers = []
            for i in range(len(entry)):
                if gate[col].iloc[i] == 0:
                    layers = []
                if entry[col].iloc[i] == 1:
                    layers = [max_hold]
                if (reentry is not None and
                        reentry[col].iloc[i] == 1 and
                        len(layers) > 0):
                    if len(layers) < max_layers:
                        layers.append(max_hold)
                pos.iloc[i, col_idx] = (len(layers) / max_layers
                                        if layers else 0.0)
                layers = [h - 1 for h in layers]
                layers = [h for h in layers if h > 0]

    return pos


def _remove_overlap(long_pos: pd.DataFrame,
                    short_pos: pd.DataFrame):
    """Zero out any bar where both long and short are active."""
    conflict    = (long_pos > 0) & (short_pos > 0)
    long_pos    = long_pos.copy()
    short_pos   = short_pos.copy()
    long_pos[conflict]  = 0.0
    short_pos[conflict] = 0.0
    return long_pos, short_pos


# -- V1: Setup Bar 9 Only ----------------------------------
def td_v1_setup9(td_setup, td_tdst,
                 pyramiding=False, max_layers=3):
    tdst_long_ready  = td_tdst['tdst_support'].notna()
    tdst_short_ready = td_tdst['tdst_resistance'].notna()

    long_entry = (
        (td_setup['bull_setup_signal'] == 1) &
        (td_tdst['tdst_long_valid'] == 1) &
        tdst_long_ready
    ).astype(int)

    short_entry = (
        (td_setup['bear_setup_signal'] == 1) &
        (td_tdst['tdst_short_valid'] == 1) &
        tdst_short_ready
    ).astype(int)

    long_pos  = carry_signal(
        long_entry, td_tdst['tdst_long_valid'],
        max_hold=63,
        pyramiding=pyramiding, max_layers=max_layers)

    short_pos = carry_signal(
        short_entry, td_tdst['tdst_short_valid'],
        max_hold=63,
        pyramiding=pyramiding, max_layers=max_layers)

    long_pos, short_pos = _remove_overlap(long_pos, short_pos)

    return long_pos.shift(1).fillna(0), \
           short_pos.shift(1).fillna(0)


# -- V2: Countdown Bar 13 Only ----------------------------
def td_v2_countdown13(td_countdown, td_tdst,
                      pyramiding=False, max_layers=3):
    tdst_long_ready  = td_tdst['tdst_support'].notna()
    tdst_short_ready = td_tdst['tdst_resistance'].notna()

    long_entry = (
        (td_countdown['bull_cd_signal'] == 1) &
        (td_tdst['tdst_long_valid'] == 1) &
        tdst_long_ready
    ).astype(int)

    short_entry = (
        (td_countdown['bear_cd_signal'] == 1) &
        (td_tdst['tdst_short_valid'] == 1) &
        tdst_short_ready
    ).astype(int)

    long_pos  = carry_signal(
        long_entry, td_tdst['tdst_long_valid'],
        max_hold=126,
        pyramiding=pyramiding,
        reentry=td_countdown['bull_cd_reentry'],
        max_layers=max_layers)

    short_pos = carry_signal(
        short_entry, td_tdst['tdst_short_valid'],
        max_hold=126,
        pyramiding=pyramiding,
        reentry=td_countdown['bear_cd_reentry'],
        max_layers=max_layers)

    long_pos, short_pos = _remove_overlap(long_pos, short_pos)

    return long_pos.shift(1).fillna(0), \
           short_pos.shift(1).fillna(0)


# -- V3: Setup Entry + Countdown Confirmation -------------
def td_v3_both(td_setup, td_countdown, td_tdst,
               pyramiding=False, max_layers=3):
    tdst_long_ready  = td_tdst['tdst_support'].notna()
    tdst_short_ready = td_tdst['tdst_resistance'].notna()

    # Long side
    setup_entry_l = (
        (td_setup['bull_setup_signal'] == 1) &
        (td_tdst['tdst_long_valid'] == 1) &
        tdst_long_ready
    ).astype(int)

    cd_entry_l = (
        (td_countdown['bull_cd_signal'] == 1) &
        (td_tdst['tdst_long_valid'] == 1) &
        tdst_long_ready
    ).astype(int)

    setup_carried_l = carry_signal(
        setup_entry_l, td_tdst['tdst_long_valid'],
        max_hold=126,
        pyramiding=pyramiding, max_layers=max_layers)

    cd_carried_l = carry_signal(
        cd_entry_l, td_tdst['tdst_long_valid'],
        max_hold=126,
        pyramiding=pyramiding,
        reentry=td_countdown['bull_cd_reentry'],
        max_layers=max_layers)

    cd_gated_l = cd_carried_l * (setup_carried_l > 0).astype(float)
    long_pos   = (setup_carried_l * 0.5 + cd_gated_l * 0.5).clip(0, 1)

    # Short side
    setup_entry_s = (
        (td_setup['bear_setup_signal'] == 1) &
        (td_tdst['tdst_short_valid'] == 1) &
        tdst_short_ready
    ).astype(int)

    cd_entry_s = (
        (td_countdown['bear_cd_signal'] == 1) &
        (td_tdst['tdst_short_valid'] == 1) &
        tdst_short_ready
    ).astype(int)

    setup_carried_s = carry_signal(
        setup_entry_s, td_tdst['tdst_short_valid'],
        max_hold=126,
        pyramiding=pyramiding, max_layers=max_layers)

    cd_carried_s = carry_signal(
        cd_entry_s, td_tdst['tdst_short_valid'],
        max_hold=126,
        pyramiding=pyramiding,
        reentry=td_countdown['bear_cd_reentry'],
        max_layers=max_layers)

    cd_gated_s = cd_carried_s * (setup_carried_s > 0).astype(float)
    short_pos  = (setup_carried_s * 0.5 + cd_gated_s * 0.5).clip(0, 1)

    long_pos, short_pos = _remove_overlap(long_pos, short_pos)

    return long_pos.shift(1).fillna(0), \
           short_pos.shift(1).fillna(0)


# -- V4: Separate Independent Alphas ----------------------
def td_v4_separate(td_setup, td_countdown, td_tdst,
                   pyramiding=False, max_layers=3):
    sl, ss = td_v1_setup9(
        td_setup, td_tdst,
        pyramiding=pyramiding, max_layers=max_layers)
    cl, cs = td_v2_countdown13(
        td_countdown, td_tdst,
        pyramiding=pyramiding, max_layers=max_layers)
    return sl, ss, cl, cs


# -- Compute all variants in BOTH pyramiding modes --------
print("Computing TD variant signals — both pyramiding modes...")

MAX_LAYERS = 3

variant_signals = {}

for pyr_mode, pyr_label in [(False, 'Standard'), (True, 'Pyramiding')]:

    v1_long,  v1_short  = td_v1_setup9(
        td_setup, td_tdst,
        pyramiding=pyr_mode, max_layers=MAX_LAYERS)

    v2_long,  v2_short  = td_v2_countdown13(
        td_countdown, td_tdst,
        pyramiding=pyr_mode, max_layers=MAX_LAYERS)

    v3_long,  v3_short  = td_v3_both(
        td_setup, td_countdown, td_tdst,
        pyramiding=pyr_mode, max_layers=MAX_LAYERS)

    v4_setup_long, v4_setup_short, \
    v4_cd_long,    v4_cd_short     = td_v4_separate(
        td_setup, td_countdown, td_tdst,
        pyramiding=pyr_mode, max_layers=MAX_LAYERS)

    variant_signals[pyr_label] = {
        'V1':       (v1_long,       v1_short),
        'V2':       (v2_long,       v2_short),
        'V3':       (v3_long,       v3_short),
        'V4_Setup': (v4_setup_long, v4_setup_short),
        'V4_CD':    (v4_cd_long,    v4_cd_short),
    }

# Expose Standard mode signals at module level
ACTIVE_MODE                         = 'Standard'
v1_long,  v1_short                  = variant_signals[ACTIVE_MODE]['V1']
v2_long,  v2_short                  = variant_signals[ACTIVE_MODE]['V2']
v3_long,  v3_short                  = variant_signals[ACTIVE_MODE]['V3']
v4_setup_long, v4_setup_short       = variant_signals[ACTIVE_MODE]['V4_Setup']
v4_cd_long,    v4_cd_short          = variant_signals[ACTIVE_MODE]['V4_CD']

# -- Signal density comparison across modes ----------------
print("\nSignal density comparison (fraction of ticker-days active):")
print(f"  {'Variant':<28s}  {'Standard':>10s}  {'Pyramiding':>10s}  {'Delta':>8s}")
print("  " + "-" * 62)
for variant in ['V1', 'V2', 'V3', 'V4_Setup', 'V4_CD']:
    for side, idx in [('Long', 0), ('Short', 1)]:
        std_sig = variant_signals['Standard'][variant][idx]
        pyr_sig = variant_signals['Pyramiding'][variant][idx]
        std_d   = std_sig.mean().mean()
        pyr_d   = pyr_sig.mean().mean()
        label   = f"{variant} {side}"
        print(f"  {label:<28s}  {std_d:>10.1%}  {pyr_d:>10.1%}"
              f"  {(pyr_d - std_d):>+8.1%}")

# -- Long/short overlap diagnostic — both modes -----------
print("\nLong/short overlap (should all be 0):")
for mode_label in ['Standard', 'Pyramiding']:
    print(f"\n  [{mode_label}]")
    sigs = variant_signals[mode_label]
    for variant in ['V1', 'V2', 'V3', 'V4_Setup', 'V4_CD']:
        lg, sh   = sigs[variant]
        overlap  = (lg * sh).sum().sum()
        flag     = '  *** WARNING ***' if overlap > 0 else ''
        print(f"    {variant:<10s}: {int(overlap):6d}{flag}")

print("\nVariant signals computed")

In [ ]:
def compute_td_signals(close, high, low, setup_count, countdown_count,
                       setup_lookback, countdown_lookback, tdst_window):
    td_setup = compute_td_setup_param(close, high, low,
                                      setup_count=setup_count,
                                      setup_lookback=setup_lookback)
    td_tdst  = compute_tdst(close, high, low,
                             td_setup['bull_setup9'],
                             td_setup['bear_setup9'],
                             setup_count=tdst_window)
    td_cd    = compute_td_countdown_param(close, high, low,
                                          td_setup['bull_setup9'],
                                          td_setup['bear_setup9'],
                                          countdown_count=countdown_count,
                                          countdown_lookback=countdown_lookback)
    return td_setup, td_tdst, td_cd, None


## Section 4 — Signal Generation

In [ ]:
# V1 — TD Setup Signal (9-bar, conventional + contrarian)
# Entry when 9 consecutive closes each < close[−4].
# Conventional (+1): bull setup -> long,  bear setup -> short.
# Contrarian  (−1): bear setup -> long,  bull setup -> short (momentum continuation).

HOLD_WINDOW = 63   # time-stop in bars

td_setup, td_tdst, _, _ = compute_td_signals(
    close, high, low,
    setup_count=SETUP_COUNT, countdown_count=13,
    setup_lookback=4, countdown_lookback=2,
    tdst_window=TDST_WINDOW,
)

v1_long_conv,  v1_short_conv  = td_v1_setup9(td_setup, td_tdst, HOLD_WINDOW, polarity=+1)
v1_long_ctr,   v1_short_ctr   = td_v1_setup9(td_setup, td_tdst, HOLD_WINDOW, polarity=-1)

for label, sig in [("Conv long", v1_long_conv), ("Conv short", v1_short_conv),
                   ("Ctr  long", v1_long_ctr),  ("Ctr  short", v1_short_ctr)]:
    print(f"V1 {label}: {int(sig.sum().sum())} signals across {len(sig.columns)} tickers")

# Combine for downstream performance cells (conventional as primary)
v_long, v_short = v1_long_conv, v1_short_conv


## Section 5 — Returns & Performance

In [ ]:
def compute_variant_returns(long_sig, short_sig,
                             returns, label):
    long_pos  = long_sig.reindex(returns.index).fillna(0)
    long_net  = apply_costs(returns, long_pos, COST_BPS)
    n_long    = long_sig.sum(axis=1).replace(0, np.nan)
    long_pnl  = (long_net * long_pos).sum(axis=1)
    long_ret  = (long_pnl / n_long).fillna(0)

    short_pos = short_sig.reindex(returns.index).fillna(0) * -1
    short_net = apply_costs(returns, short_pos, COST_BPS)
    n_short   = short_sig.sum(axis=1).replace(0, np.nan)
    short_pnl = (short_net * short_pos).sum(axis=1)
    short_ret = (short_pnl / n_short).fillna(0)

    combined  = ((long_ret + short_ret) / 2).rename(label)
    return long_ret.rename(f'{label}_Long'), \
           short_ret.rename(f'{label}_Short'), \
           combined


print("Computing variant returns — both pyramiding modes...")

# Compute returns for both modes; store in dict
all_variants_by_mode = {}
perf_by_mode         = {}

for mode_label in ['Standard', 'Pyramiding']:
    sigs = variant_signals[mode_label]

    _vl = {}   # combined return streams per variant
    _ll = {}   # long-only
    _sl = {}   # short-only

    v1_l,  v1_s,  v1_c  = compute_variant_returns(*sigs['V1'],       returns, 'V1_Setup9')
    v2_l,  v2_s,  v2_c  = compute_variant_returns(*sigs['V2'],       returns, 'V2_CD13')
    v3_l,  v3_s,  v3_c  = compute_variant_returns(*sigs['V3'],       returns, 'V3_Both')
    v4_sl, v4_ss, v4_sc = compute_variant_returns(*sigs['V4_Setup'],  returns, 'V4_Setup')
    v4_cl, v4_cs, v4_cc = compute_variant_returns(*sigs['V4_CD'],     returns, 'V4_CD')

    v4_combined = ((v4_sl + v4_ss + v4_cl + v4_cs) / 4
                   ).rename('V4_Separate')

    av = pd.DataFrame({
        'V1_Setup9':   v1_c,
        'V2_CD13':     v2_c,
        'V3_Both':     v3_c,
        'V4_Separate': v4_combined,
    })

    all_variants_by_mode[mode_label] = av

    perf_rows = [performance_summary(av[col], name=col)
                 for col in av.columns]
    perf_by_mode[mode_label] = pd.DataFrame(perf_rows)

    # Store long/short streams for downstream use
    if mode_label == ACTIVE_MODE:
        v1_long_ret, v1_short_ret   = v1_l,  v1_s
        v2_long_ret, v2_short_ret   = v2_l,  v2_s
        v3_long_ret, v3_short_ret   = v3_l,  v3_s
        v4_setup_long_ret           = v4_sl
        v4_setup_short_ret          = v4_ss
        v4_cd_long_ret              = v4_cl
        v4_cd_short_ret             = v4_cs
        all_variants                = av
        perf_table                  = perf_by_mode[mode_label]

# -- Performance comparison table -------------------------
print("\nCombined Long+Short Performance — Standard vs Pyramiding:")
print("=" * 70)

metrics = ['Ann. Return', 'Sharpe', 'Max DD', 'Calmar']
for metric in metrics:
    print(f"\n  {metric}:")
    print(f"  {'Variant':<16s}  {'Standard':>10s}  "
          f"{'Pyramiding':>10s}  {'Delta':>8s}")
    print("  " + "-" * 50)
    for variant in ['V1_Setup9', 'V2_CD13', 'V3_Both', 'V4_Separate']:
        std_val = perf_by_mode['Standard'].loc[variant, metric]
        pyr_val = perf_by_mode['Pyramiding'].loc[variant, metric]
        delta   = pyr_val - std_val
        fmt     = '.1%' if metric in ['Ann. Return', 'Max DD'] else '.2f'
        print(f"  {variant:<16s}  {std_val:>10{fmt}}  "
              f"{pyr_val:>10{fmt}}  {delta:>+8{fmt}}")

print(f"\n  (Downstream cells use: ACTIVE_MODE = '{ACTIVE_MODE}')")
print("\nReturns computed")




## Section 6 — Per-Ticker Breakdown

In [ ]:
print("Computing per-ticker breakdown...")


def per_ticker_sharpe(long_sig, short_sig, returns, label,
                      mode='active_only'):
    """
    Sharpe ratio per ticker for one variant.

    mode='active_only'  : Sharpe on active days only
                          — pure signal quality
    mode='capital'      : Sharpe scaled by active fraction
                          — capital efficiency
    """
    sharpes  = {}
    activity = {}

    for ticker in TICKERS:
        lp = long_sig[ticker].reindex(returns.index).fillna(0)
        sp = short_sig[ticker].reindex(returns.index).fillna(0) * -1

        ln = apply_costs(returns[[ticker]],
                         lp.to_frame(), COST_BPS)[ticker]
        sn = apply_costs(returns[[ticker]],
                         sp.to_frame(), COST_BPS)[ticker]

        lr = (ln * lp).fillna(0)
        sr = (sn * sp).fillna(0)
        cr = (lr + sr) / 2

        active_frac      = (cr != 0).mean()
        activity[ticker] = active_frac

        if mode == 'active_only':
            r          = cr[cr != 0].dropna()
            ann_factor = np.sqrt(252)
        else:
            r          = cr.dropna()
            ann_factor = np.sqrt(252 * active_frac)

        mu  = r.mean()
        std = r.std()
        sharpes[ticker] = (mu / std * ann_factor
                           if std > 0 and len(r) > 30
                           else np.nan)

    return pd.Series(sharpes, name=label), \
           pd.Series(activity, name=label)


variant_specs = [
    ('V1_Setup9',  *variant_signals[ACTIVE_MODE]['V1']),
    ('V2_CD13',    *variant_signals[ACTIVE_MODE]['V2']),
    ('V3_Both',    *variant_signals[ACTIVE_MODE]['V3']),
    ('V4_Setup',   *variant_signals[ACTIVE_MODE]['V4_Setup']),
    ('V4_CD',      *variant_signals[ACTIVE_MODE]['V4_CD']),
]

results     = {}
activity_df = None

for mode in ['active_only', 'capital']:
    sharpe_rows   = {}
    activity_rows = {}
    for variant, ls, ss in variant_specs:
        sh, act              = per_ticker_sharpe(
            ls, ss, returns, variant, mode=mode)
        sharpe_rows[variant]   = sh
        activity_rows[variant] = act
    results[mode] = pd.DataFrame(sharpe_rows)
    if activity_df is None:
        activity_df = pd.DataFrame(activity_rows)

print("\nPer-Ticker Sharpe — Signal Quality (Active Days Only):")
print("=" * 60)
print(results['active_only'].round(2).to_string())

print("\nPer-Ticker Sharpe — Capital Efficiency (All Days):")
print("=" * 60)
print(results['capital'].round(2).to_string())

print("\nActivity Rate (fraction of days with open position):")
print("=" * 60)
print(activity_df.round(3).to_string())

print("\nDivergence (Signal Quality minus Capital Efficiency):")
print("(large gap = good signal, low activity)")
print("=" * 60)
divergence = (results['active_only'] - results['capital']).round(2)
print(divergence.to_string())

# Use active_only for the dashboard heatmap
ticker_sharpe_df = results['active_only']

print("\nPer-ticker breakdown complete")




## Section 7 — Parameter Sensitivity

In [ ]:
SETUP_RANGE      = [9]
COUNTDOWN_RANGE  = [8, 9, 10, 11, 12, 13]
SWEEP_CANONICAL  = (9, 13)

print("=" * 60)
print("Cell 8: Parameter Sweep")
print(f"  Setup count    : {SETUP_RANGE}")
print(f"  Countdown count: {COUNTDOWN_RANGE}")
print(f"  Total combos   : {len(SETUP_RANGE) * len(COUNTDOWN_RANGE)}")
print(f"  Variants       : V1, V2, V3, V4 (Standard mode)")
print(f"  Sides          : Long and Short tracked separately")
print("=" * 60)

# ── Combo return helper ────────────────────────────────────
# Returns (long_ret, short_ret, combined_ret) as separate Series.
def _combo_ret(ls, ss):
    lp  = ls.reindex(returns.index).fillna(0)
    sp  = ss.reindex(returns.index).fillna(0)
    ln  = apply_costs(returns,  lp, COST_BPS)
    sn  = apply_costs(-returns, sp, COST_BPS)
    nl  = lp.sum(axis=1).replace(0, np.nan)
    ns  = sp.sum(axis=1).replace(0, np.nan)
    lr  = (ln * lp).sum(axis=1) / nl
    sr  = (sn * sp).sum(axis=1) / ns
    lr  = lr.fillna(0)
    sr  = sr.fillna(0)
    return lr, sr, (lr + sr) / 2

# ── Storage ───────────────────────────────────────────────
# sweep_results[(sc, cc)][vname] = {
#     'long': Series, 'short': Series, 'combined': Series }
sweep_results  = {}
sweep_sharpes  = {}   # [(sc,cc)] -> {vname_Long, vname_Short, vname_Combined}
total_combos   = len(SETUP_RANGE) * len(COUNTDOWN_RANGE)
combo_idx      = 0
t_sweep_start  = time.time()

for sc in SETUP_RANGE:
    for cc in COUNTDOWN_RANGE:
        combo_idx += 1
        elapsed   = time.time() - t_sweep_start
        eta       = (elapsed / combo_idx) * (total_combos - combo_idx)
        print(f"  [{combo_idx:3d}/{total_combos}]  "
              f"setup={sc:2d}  cd={cc:2d}  "
              f"elapsed={elapsed:5.1f}s  eta={eta:5.1f}s",
              end='\r')

        # -- Run engine -----------------------------------
        _setup = compute_td_setup_param(
            close, high, low,
            setup_count=sc, setup_lookback=4)

        _tdst = compute_tdst(
            close, high, low,
            _setup['bull_setup_signal'],
            _setup['bear_setup_signal'],
            setup_count=sc)

        _cd = compute_td_countdown_param(
            close, high, low,
            _setup['bull_setup_signal'],
            _setup['bear_setup_signal'],
            countdown_count=cc, countdown_lookback=2)

        _v1l,  _v1s              = td_v1_setup9(
            _setup, _tdst, pyramiding=False)
        _v2l,  _v2s              = td_v2_countdown13(
            _cd, _tdst, pyramiding=False)
        _v3l,  _v3s              = td_v3_both(
            _setup, _cd, _tdst, pyramiding=False)
        _v4sl, _v4ss, _v4cl, _v4cs = td_v4_separate(
            _setup, _cd, _tdst, pyramiding=False)

        # -- Store long, short, combined per variant ------
        v1_l,  v1_s,  v1_c  = _combo_ret(_v1l,  _v1s)
        v2_l,  v2_s,  v2_c  = _combo_ret(_v2l,  _v2s)
        v3_l,  v3_s,  v3_c  = _combo_ret(_v3l,  _v3s)
        v4s_l, v4s_s, v4s_c = _combo_ret(_v4sl, _v4ss)
        v4c_l, v4c_s, v4c_c = _combo_ret(_v4cl, _v4cs)

        # V4 combined = mean of setup alpha + CD alpha
        v4_l = (v4s_l + v4c_l) / 2
        v4_s = (v4s_s + v4c_s) / 2
        v4_c = (v4_l  + v4_s)  / 2

        combo_rets = {
            'V1_Setup9':   {'long': v1_l,  'short': v1_s,  'combined': v1_c},
            'V2_CD13':     {'long': v2_l,  'short': v2_s,  'combined': v2_c},
            'V3_Both':     {'long': v3_l,  'short': v3_s,  'combined': v3_c},
            'V4_Separate': {'long': v4_l,  'short': v4_s,  'combined': v4_c},
        }

        sweep_results[(sc, cc)] = combo_rets

        # -- Sharpe for long, short, combined -------------
        sharpe_row = {}
        for vname, sides in combo_rets.items():
            for side, r in sides.items():
                r_   = r.dropna()
                mu   = r_.mean()
                std  = r_.std()
                key  = f'{vname}_{side}'
                sharpe_row[key] = (
                    mu / std * np.sqrt(252)
                    if std > 0 else np.nan)
        sweep_sharpes[(sc, cc)] = sharpe_row

elapsed_total = time.time() - t_sweep_start
print(f"\n\nSweep complete in {elapsed_total:.1f}s")

# ── Build Sharpe matrices ─────────────────────────────────
sc_vals       = SETUP_RANGE
cc_vals       = COUNTDOWN_RANGE
variant_names = ['V1_Setup9', 'V2_CD13', 'V3_Both', 'V4_Separate']
side_names    = ['long', 'short', 'combined']

# sharpe_matrices[vname][side] = DataFrame(index=sc_vals, columns=cc_vals)
sharpe_matrices = {}
for vname in variant_names:
    sharpe_matrices[vname] = {}
    for side in side_names:
        mat = pd.DataFrame(index=sc_vals, columns=cc_vals, dtype=float)
        for sc in sc_vals:
            for cc in cc_vals:
                key = f'{vname}_{side}'
                mat.loc[sc, cc] = sweep_sharpes[(sc, cc)][key]
        sharpe_matrices[vname][side] = mat

# ── Mask unreachable cells (cc <= sc) for CD-dependent variants
for vname in variant_names:
    if vname == 'V1_Setup9':
        continue
    for side in side_names:
        mat = sharpe_matrices[vname][side]
        for sc in sc_vals:
            for cc in cc_vals:
                if cc <= sc:
                    mat.loc[sc, cc] = np.nan
        sharpe_matrices[vname][side] = mat

# ── Best configurations per variant (combined Sharpe) ─────
print("\nTop 5 configurations by Combined Sharpe:")
print("=" * 60)
for vname in variant_names:
    mat  = sharpe_matrices[vname]['combined']
    flat = mat.stack().dropna().sort_values(ascending=False)
    print(f"\n  {vname}:")
    print(f"  {'Setup':>6s}  {'CD':>6s}  {'Combined':>10s}  "
          f"{'Long':>8s}  {'Short':>8s}")
    for (sc, cc), sharpe in flat.head(5).items():
        l_sh  = sharpe_matrices[vname]['long'].loc[sc, cc]
        s_sh  = sharpe_matrices[vname]['short'].loc[sc, cc]
        canon = ' ← canonical' if (sc, cc) == SWEEP_CANONICAL else ''
        print(f"  {sc:>6d}  {cc:>6d}  {sharpe:>10.3f}  "
              f"{l_sh:>8.3f}  {s_sh:>8.3f}{canon}")

# ── Figure 1: Sharpe heatmaps — long / short / combined ───
# One figure per variant, three rows (long, short, combined).
print("\nPlotting Sharpe heatmaps...")
from matplotlib.patches import Rectangle

for vname in variant_names:
    fig_heat, axes_heat = plt.subplots(
        1, 3, figsize=(18, 4 * len(sc_vals)))
    axes_heat = axes_heat.flatten()

    for ax, side in zip(axes_heat, side_names):
        mat  = sharpe_matrices[vname][side].astype(float)
        vals = mat.values
        vmin = np.nanpercentile(vals, 5)
        vmax = np.nanpercentile(vals, 95)

        ax.set_facecolor('#cccccc')

        im = ax.imshow(
            vals,
            aspect='auto',
            cmap='RdYlGn',
            vmin=vmin, vmax=vmax,
            interpolation='nearest',
            origin='upper')

        for i, sc in enumerate(sc_vals):
            for j, cc in enumerate(cc_vals):
                val = mat.loc[sc, cc]
                if not np.isnan(val):
                    is_canon = (sc, cc) == SWEEP_CANONICAL
                    ax.text(j, i, f'{val:.2f}',
                            ha='center', va='center',
                            fontsize=8,
                            fontweight='bold' if is_canon else 'normal',
                            color='white'
                            if abs(val - (vmin + vmax) / 2) > (vmax - vmin) * 0.3
                            else 'black')

        if (SWEEP_CANONICAL[0] in sc_vals and
                SWEEP_CANONICAL[1] in cc_vals):
            ci = sc_vals.index(SWEEP_CANONICAL[0])
            cj = cc_vals.index(SWEEP_CANONICAL[1])
            rect = Rectangle(
                (cj - 0.5, ci - 0.5), 1, 1,
                linewidth=2.5, edgecolor='navy',
                facecolor='none', zorder=5)
            ax.add_patch(rect)

        ax.set_xticks(range(len(cc_vals)))
        ax.set_xticklabels(cc_vals, fontsize=9)
        ax.set_yticks(range(len(sc_vals)))
        ax.set_yticklabels(sc_vals, fontsize=9)
        ax.set_xlabel('Countdown Count', fontsize=10)
        ax.set_ylabel('Setup Count', fontsize=10)
        ax.set_title(f'{side.capitalize()} Sharpe',
                     fontweight='bold', fontsize=11)
        plt.colorbar(im, ax=ax, shrink=0.8, label='Sharpe')

    fig_heat.suptitle(
        f'{vname} — Sharpe Heatmaps: Long | Short | Combined\n'
        f'Setup {SETUP_RANGE} × Countdown {COUNTDOWN_RANGE}  |  '
        f'navy = canonical 9/13  |  grey = unreachable',
        fontsize=12, fontweight='bold')
    fig_heat.tight_layout(rect=[0, 0, 1, 0.93])
    fname = f'td_sweep_heatmap_{vname.lower()}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {fname}")

# ── Figure 2: Equity curves — long / short / combined ─────
print("\nPlotting equity curve grids...")
import matplotlib.cm as cm
import matplotlib.colors as mcolors

for vname in variant_names:
    fig_eq, axes_eq = plt.subplots(
        3, 1, figsize=(16, 18), sharex=True)

    for ax, side in zip(axes_eq, side_names):
        mat    = sharpe_matrices[vname][side]
        all_sh = mat.values.flatten()
        sh_min = np.nanpercentile(all_sh, 5)
        sh_max = np.nanpercentile(all_sh, 95)
        norm     = mcolors.Normalize(vmin=sh_min, vmax=sh_max)
        cmap_obj = cm.get_cmap('RdYlGn')

        for sc in sc_vals:
            for cc in cc_vals:
                if vname != 'V1_Setup9' and cc <= sc:
                    continue
                if (sc, cc) == SWEEP_CANONICAL:
                    continue
                r      = sweep_results[(sc, cc)][vname][side]
                cum    = (1 + r).cumprod()
                sharpe = sharpe_matrices[vname][side].loc[sc, cc]
                color  = (cmap_obj(norm(sharpe))
                          if not np.isnan(sharpe)
                          else (0.7, 0.7, 0.7, 0.5))
                ax.plot(cum.index, cum.values,
                        color=color, linewidth=0.8,
                        alpha=0.65, zorder=2,
                        label=f'sc={sc} cd={cc}')

        # Canonical on top
        if SWEEP_CANONICAL in sweep_results:
            r_canon   = sweep_results[SWEEP_CANONICAL][vname][side]
            cum_canon = (1 + r_canon).cumprod()
            ax.plot(cum_canon.index, cum_canon.values,
                    color='navy', linewidth=2.5, zorder=5,
                    label=f'Canonical '
                          f'({SWEEP_CANONICAL[0]},{SWEEP_CANONICAL[1]})')

        # SPY B&H — only meaningful reference for long side
        if side == 'long':
            ax.plot(spy_cum.index, spy_cum.values,
                    color='#555555', linewidth=1.8,
                    linestyle='--', label='SPY B&H', zorder=4)

        ax.axhline(1, color='grey', linewidth=0.8, linestyle=':')

        sm = cm.ScalarMappable(cmap=cmap_obj, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, label='Sharpe',
                     shrink=0.6, pad=0.01)

        ax.set_ylabel('Cumulative Return')
        ax.set_title(f'{side.capitalize()} Side',
                     fontweight='bold', fontsize=10)
        ax.legend(fontsize=8, loc='upper left',
                  ncol=3, framealpha=0.7)
        ax.grid(True, alpha=0.25)

    fig_eq.suptitle(
        f'{vname} — Equity Curves: Long | Short | Combined\n'
        f'Setup {SETUP_RANGE}  Countdown {COUNTDOWN_RANGE}  '
        f'({total_combos} combos)  |  '
        f'Navy = canonical  |  Colour = Sharpe',
        fontsize=12, fontweight='bold')
    fig_eq.tight_layout(rect=[0, 0, 1, 0.96])
    fname = f'td_sweep_equity_{vname.lower()}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {fname}")

# ── Figure 3: Sharpe distribution — long / short / combined
print("\nPlotting Sharpe distributions...")

for vname in variant_names:
    fig_dist, axes_dist = plt.subplots(
        1, 3, figsize=(18, 5), sharey=False)

    for ax, side in zip(axes_dist, side_names):
        mat    = sharpe_matrices[vname][side]
        all_sh = mat.values.flatten()
        all_sh = all_sh[~np.isnan(all_sh)]

        canon_sh = (
            sweep_sharpes[SWEEP_CANONICAL][f'{vname}_{side}']
            if SWEEP_CANONICAL in sweep_sharpes else np.nan)

        ax.hist(all_sh, bins=max(5, len(all_sh) // 2),
                color='steelblue', edgecolor='white',
                alpha=0.8, label='All combos')
        if not np.isnan(canon_sh):
            ax.axvline(canon_sh, color='navy', linewidth=2,
                       linestyle='--',
                       label=f'Canonical ({canon_sh:.2f})')
        ax.axvline(np.nanmean(all_sh), color='darkorange',
                   linewidth=1.5, linestyle=':',
                   label=f'Mean ({np.nanmean(all_sh):.2f})')
        ax.set_title(f'{side.capitalize()}',
                     fontweight='bold', fontsize=10)
        ax.set_xlabel('Sharpe Ratio')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')

        if not np.isnan(canon_sh) and len(all_sh) > 1:
            pct = (all_sh < canon_sh).mean() * 100
            ax.text(0.97, 0.97,
                    f'Canonical\npercentile:\n{pct:.0f}th',
                    transform=ax.transAxes,
                    ha='right', va='top', fontsize=8,
                    bbox=dict(boxstyle='round',
                              facecolor='lightyellow',
                              alpha=0.8))

    fig_dist.suptitle(
        f'{vname} — Sharpe Distribution: Long | Short | Combined\n'
        f'Canonical (9,13) vs all {total_combos} configurations',
        fontsize=12, fontweight='bold')
    fig_dist.tight_layout(rect=[0, 0, 1, 0.93])
    fname = f'td_sweep_dist_{vname.lower()}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {fname}")

print("\nCell 8 complete — files saved:")
for vname in variant_names:
    print(f"  td_sweep_heatmap_{vname.lower()}.png")
    print(f"  td_sweep_equity_{vname.lower()}.png")
    print(f"  td_sweep_dist_{vname.lower()}.png")

## Section 8 — Representative Equity Curves

In [ ]:
print("=" * 60)
print("Cell 9: Representative Equity Curves")
print("  Method : Sharpe-weighted mean of daily returns")
print("  Band   : 25th / 75th percentile across all combos")
print(f"  Combos : {len(sc_vals) * len(cc_vals)}  "
      f"(setup={sc_vals}  cd={cc_vals})")
print("=" * 60)

# ── Build weight matrices and representative returns ───────
representative = {}

for vname in variant_names:

    # -- Collect all daily return Series ------------------
    all_rets = {}
    for sc in sc_vals:
        for cc in cc_vals:
            all_rets[(sc, cc)] = sweep_results[(sc, cc)][vname]

    ret_df = pd.DataFrame(all_rets).reindex(returns.index).fillna(0)

    # -- Compute Sharpe-based weights ---------------------
    raw_weights = {}
    for key in all_rets:
        sh = sweep_sharpes[key][vname]
        raw_weights[key] = max(0.0, sh) if not np.isnan(sh) else 0.0

    total_w = sum(raw_weights.values())
    if total_w == 0:
        weights = {k: 1.0 / len(raw_weights) for k in raw_weights}
    else:
        weights = {k: v / total_w for k, v in raw_weights.items()}

    weight_series = pd.Series(weights)

    # -- Weighted mean daily return -----------------------
    weight_arr = np.array([weights[k] for k in ret_df.columns])
    rep_ret    = ret_df.values @ weight_arr
    rep_ret    = pd.Series(rep_ret, index=ret_df.index,
                           name=f'{vname}_rep')

    # -- Cumulative curves for all combos (for band) ------
    cum_df  = (1 + ret_df).cumprod()

    # -- Percentile bands ---------------------------------
    band_25 = cum_df.quantile(0.25, axis=1)
    band_75 = cum_df.quantile(0.75, axis=1)
    band_10 = cum_df.quantile(0.10, axis=1)
    band_90 = cum_df.quantile(0.90, axis=1)

    # -- Representative cumulative curve ------------------
    rep_cum = (1 + rep_ret).cumprod()

    # -- Canonical curve (guarded) ------------------------
    if SWEEP_CANONICAL in sweep_results:
        canon_ret = sweep_results[SWEEP_CANONICAL][vname]
        canon_cum = (1 + canon_ret.reindex(
            returns.index).fillna(0)).cumprod()
    else:
        canon_cum = None

    # -- Effective N --------------------------------------
    uniform_w   = 1.0 / len(weights)
    effective_n = sum(1 for w in weights.values()
                      if w > uniform_w * 0.5)

    rep_perf = performance_summary(rep_ret, name=vname)

    representative[vname] = {
        'rep_ret':     rep_ret,
        'rep_cum':     rep_cum,
        'canon_cum':   canon_cum,
        'band_25':     band_25,
        'band_75':     band_75,
        'band_10':     band_10,
        'band_90':     band_90,
        'weights':     weight_series,
        'effective_n': effective_n,
        'perf':        rep_perf,
    }

n_combos = len(sc_vals) * len(cc_vals)

# ── Print summary table ────────────────────────────────────
print("\nRepresentative Curve Performance (Sharpe-Weighted):")
print("=" * 60)
rep_perf_rows = [representative[v]['perf'] for v in variant_names]
rep_perf_df   = pd.DataFrame(rep_perf_rows)
print(rep_perf_df.round(3).to_string())

print("\nEffective parameter count (configs with meaningful weight):")
for vname in variant_names:
    en = representative[vname]['effective_n']
    print(f"  {vname:<16s}: {en:3d} / {n_combos} combos")

# ── Figure 1: Representative curves — all four variants ───
print("\nPlotting representative equity curves...")

fig_rep, axes_rep = plt.subplots(2, 2, figsize=(20, 14), sharex=True)
axes_rep = axes_rep.flatten()

for ax, vname in zip(axes_rep, variant_names):
    d = representative[vname]

    ax.fill_between(
        d['band_25'].index,
        d['band_25'].values,
        d['band_75'].values,
        color='steelblue', alpha=0.18,
        label='25th–75th pct band')

    ax.fill_between(
        d['band_10'].index,
        d['band_10'].values,
        d['band_90'].values,
        color='steelblue', alpha=0.08,
        label='10th–90th pct band')

    for key in sweep_results:
        r   = sweep_results[key][vname]
        cum = (1 + r.reindex(returns.index).fillna(0)).cumprod()
        ax.plot(cum.index, cum.values,
                color='lightsteelblue', linewidth=0.3,
                alpha=0.25, zorder=1)

    ax.plot(spy_cum.index, spy_cum.values,
            color='#555555', linewidth=1.6,
            linestyle='--', label='SPY B&H', zorder=3)

    if d['canon_cum'] is not None:
        ax.plot(d['canon_cum'].index, d['canon_cum'].values,
                color='navy', linewidth=2.0,
                linestyle='-.', zorder=4,
                label=f'Canonical '
                      f'({SWEEP_CANONICAL[0]},{SWEEP_CANONICAL[1]})')

    ax.plot(d['rep_cum'].index, d['rep_cum'].values,
            color='crimson', linewidth=2.5,
            zorder=5,
            label=f'Sharpe-Weighted Mean '
                  f'(N_eff={d["effective_n"]})')

    ax.axhline(1, color='grey', linewidth=0.7, linestyle=':')

    sh  = d['perf']['Sharpe']
    mdd = d['perf']['Max DD']
    ax.set_title(
        f'{vname}\n'
        f'Rep. Sharpe={sh:.2f}  Max DD={mdd:.1%}  '
        f'Effective configs={d["effective_n"]}/{n_combos}',
        fontweight='bold', fontsize=10)
    ax.set_ylabel('Cumulative Return')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.25)

fig_rep.suptitle(
    'TD Sequential — Representative Equity Curves\n'
    f'Sharpe-Weighted Mean  |  25/75 & 10/90 Bands  |  '
    f'Setup {sc_vals}  Countdown {cc_vals}  ({n_combos} combos)',
    fontsize=13, fontweight='bold')
fig_rep.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('td_representative_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: td_representative_curves.png")

# ── Figure 2: Weight distribution heatmaps ────────────────
print("\nPlotting weight distribution heatmaps...")

fig_wt, axes_wt = plt.subplots(2, 2, figsize=(14, 10))
axes_wt = axes_wt.flatten()

for ax, vname in zip(axes_wt, variant_names):
    d      = representative[vname]
    wt_mat = pd.DataFrame(
        index=sc_vals, columns=cc_vals, dtype=float)
    for sc in sc_vals:
        for cc in cc_vals:
            wt_mat.loc[sc, cc] = d['weights'].get((sc, cc), 0.0)

    im = ax.imshow(
        wt_mat.values,
        aspect='auto',
        cmap='YlOrRd',
        vmin=0,
        interpolation='nearest',
        origin='upper')

    for i, sc in enumerate(sc_vals):
        for j, cc in enumerate(cc_vals):
            val = wt_mat.loc[sc, cc]
            ax.text(j, i, f'{val:.3f}',
                    ha='center', va='center',
                    fontsize=8,
                    color='white'
                    if val > wt_mat.values.max() * 0.6
                    else 'black')

    if (SWEEP_CANONICAL[0] in sc_vals and
            SWEEP_CANONICAL[1] in cc_vals):
        from matplotlib.patches import Rectangle
        ci = sc_vals.index(SWEEP_CANONICAL[0])
        cj = cc_vals.index(SWEEP_CANONICAL[1])
        rect = Rectangle(
            (cj - 0.5, ci - 0.5), 1, 1,
            linewidth=2.5, edgecolor='navy',
            facecolor='none', zorder=5)
        ax.add_patch(rect)

    ax.set_xticks(range(len(cc_vals)))
    ax.set_xticklabels(cc_vals, fontsize=9)
    ax.set_yticks(range(len(sc_vals)))
    ax.set_yticklabels(sc_vals, fontsize=9)
    ax.set_xlabel('Countdown Count', fontsize=10)
    ax.set_ylabel('Setup Count', fontsize=10)
    ax.set_title(
        f'{vname} — Sharpe Weight per Config\n'
        f'(navy border = canonical 9/13  |  '
        f'darker = higher weight)',
        fontweight='bold', fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8, label='Normalised Weight')

fig_wt.suptitle(
    'TD Sequential — Parameter Weight Distribution\n'
    'Which configurations dominate the representative curve?',
    fontsize=13, fontweight='bold')
fig_wt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('td_representative_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: td_representative_weights.png")

# ── Figure 3: Representative vs Canonical comparison ──────
print("\nPlotting representative vs canonical comparison...")

fig_comp, ax_comp = plt.subplots(figsize=(18, 7))

comp_colors = {
    'V1_Setup9':   ('#1f77b4', '#aec7e8'),
    'V2_CD13':     ('#ff7f0e', '#ffbb78'),
    'V3_Both':     ('#2ca02c', '#98df8a'),
    'V4_Separate': ('#d62728', '#ff9896'),
}

for vname in variant_names:
    d         = representative[vname]
    rep_color = comp_colors[vname][0]
    can_color = comp_colors[vname][1]

    ax_comp.plot(
        d['rep_cum'].index, d['rep_cum'].values,
        color=rep_color, linewidth=2.2,
        label=f'{vname} weighted', zorder=4)

    if d['canon_cum'] is not None:
        ax_comp.plot(
            d['canon_cum'].index, d['canon_cum'].values,
            color=can_color, linewidth=1.2,
            linestyle='--', zorder=3,
            label=f'{vname} canonical')

ax_comp.plot(spy_cum.index, spy_cum.values,
             color='#555555', linewidth=1.8,
             linestyle=':', label='SPY B&H', zorder=5)
ax_comp.axhline(1, color='grey', linewidth=0.7, linestyle=':')

ax_comp.set_title(
    f'Representative (Sharpe-Weighted) vs Canonical '
    f'({SWEEP_CANONICAL[0]},{SWEEP_CANONICAL[1]}) — All Variants\n'
    'Solid = weighted representative  |  Dashed = canonical  |  '
    'Dotted = SPY B&H',
    fontweight='bold', fontsize=11)
ax_comp.set_ylabel('Cumulative Return')
ax_comp.legend(fontsize=9, ncol=3, loc='upper left')
ax_comp.grid(True, alpha=0.25)

fig_comp.tight_layout()
plt.savefig('td_representative_vs_canonical.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: td_representative_vs_canonical.png")

print("\nCell 9 complete — files saved:")
print("  td_representative_curves.png")
print("  td_representative_weights.png")
print("  td_representative_vs_canonical.png")

In [ ]:
def plot_ticker_equity_grid(signal_ret_series, label, side, tickers,
                             close, returns, cost_bps=COST_BPS):
    """
    For each ticker plot the alpha equity curve vs ticker B&H.

    signal_ret_series : dict {ticker: daily_return_Series}
                        or a single combined Series keyed by ticker
    label             : e.g. 'V1_Setup9'
    side              : 'Long' or 'Short'
    """
    n       = len(tickers)
    ncols   = int(np.ceil(np.sqrt(n)))
    nrows   = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 5, nrows * 4))
    axes = np.array(axes).flatten()

    for idx, ticker in enumerate(tickers):
        ax = axes[idx]

        # -- Alpha equity curve ---------------------------
        alpha_r = signal_ret_series[ticker].fillna(0)
        alpha_cum = (1 + alpha_r).cumprod()

        # -- Ticker B&H -----------------------------------
        tkr_r   = close[ticker].pct_change().fillna(0)
        tkr_r   = tkr_r.reindex(alpha_r.index).fillna(0)
        tkr_cum = (1 + tkr_r).cumprod()

        # -- Stats ----------------------------------------
        alpha_r_ = alpha_r[alpha_r != 0].dropna()
        mu       = alpha_r_.mean()
        std      = alpha_r_.std()
        sharpe   = mu / std * np.sqrt(252) if std > 0 else np.nan
        cum_max  = alpha_cum.cummax()
        mdd      = (alpha_cum / cum_max - 1).min()

        # -- Plot -----------------------------------------
        ax.plot(alpha_cum.index, alpha_cum.values,
                color='crimson', linewidth=1.5,
                label=f'{label} {side}', zorder=3)
        ax.plot(tkr_cum.index, tkr_cum.values,
                color='#555555', linewidth=1.2,
                linestyle='--', label=f'{ticker} B&H', zorder=2)
        ax.axhline(1, color='grey', linewidth=0.6, linestyle=':')

        sh_str  = f'{sharpe:.2f}' if not np.isnan(sharpe) else 'n/a'
        ax.set_title(f'{ticker}  |  Sharpe={sh_str}  MDD={mdd:.1%}',
                     fontsize=9, fontweight='bold')
        ax.legend(fontsize=7, loc='upper left')
        ax.grid(True, alpha=0.25)
        ax.set_ylabel('Cum. Return', fontsize=8)

    # Hide unused subplots
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(
        f'{label} — {side} Side  |  Alpha vs Ticker B&H\n'
        f'Sharpe and MDD computed on active days only',
        fontsize=13, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fname = f'td_ticker_{label.lower()}_{side.lower()}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {fname}")


def extract_ticker_rets(long_sig, short_sig, side, returns, cost_bps):
    """
    Extract per-ticker return Series for one side.
    Returns dict {ticker: Series}.
    """
    ticker_rets = {}
    for ticker in returns.columns:
        if side == 'Long':
            pos = long_sig[ticker].reindex(returns.index).fillna(0)
            r   = apply_costs(returns[[ticker]], pos.to_frame(),
                              cost_bps)[ticker]
            ticker_rets[ticker] = (r * pos).fillna(0)
        else:
            pos = short_sig[ticker].reindex(returns.index).fillna(0)
            r   = apply_costs((-returns)[[ticker]], pos.to_frame(),
                              cost_bps)[ticker]
            ticker_rets[ticker] = (r * pos).fillna(0)
    return ticker_rets

print("Ticker equity grid helpers loaded.")

## Section 9 — Export Signals for Swarm

In [ ]:
# ── Export signals ─────────────────────────────────────────
import os, pandas as pd

_longs          = v_long.stack().reset_index()
_longs.columns  = ['date', 'ticker', 'active']
_longs          = _longs[_longs['active'] > 0][['date', 'ticker']].copy()
_longs['direction'] = 1

_shorts         = v_short.stack().reset_index()
_shorts.columns = ['date', 'ticker', 'active']
_shorts         = _shorts[_shorts['active'] > 0][['date', 'ticker']].copy()
_shorts['direction'] = -1

signals_df = pd.concat([_longs, _shorts], ignore_index=True)
signals_df['alpha_name'] = 'alpha_td_setup'
signals_df['date'] = pd.to_datetime(signals_df['date']).dt.strftime('%Y-%m-%d')
signals_df = signals_df.sort_values(['date', 'ticker']).reset_index(drop=True)

# Primary: write locally so GitHub Actions can commit it
os.makedirs('outputs', exist_ok=True)
signals_df.to_csv('outputs/signals_latest.csv', index=False)
print(f'Signals written  -> outputs/signals_latest.csv  ({len(signals_df)} rows)')

# Mirror to Drive if running in Colab
if DRIVE_BASE:
    _drive_dir = os.path.join(DRIVE_BASE, 'alpha_td_setup')
    os.makedirs(_drive_dir, exist_ok=True)
    signals_df.to_csv(os.path.join(_drive_dir, 'signals_latest.csv'), index=False)
    print(f'Signals mirrored -> {_drive_dir}/signals_latest.csv')

print(signals_df.tail())
